In [0]:
pip install faker

In [0]:
pip install geopy

In [0]:
%restart_python

In [0]:
import random
from geopy.geocoders import Nominatim
from faker import Faker


geolocator = Nominatim(user_agent="customer-generator")


def get_location(country, state=None, city=None, district=None):
    """Find the requested district."""
    query = ", ".join(
        part for part in [district, city, state, country]
        if part
    )

    location = geolocator.geocode(
        query,
        addressdetails=True,
        exactly_one=True
    )

    if not location:
        raise ValueError(f"Location not found: {query}")

    return location


def get_address(location, district):
    """Find a random address within the district's approximate area."""

    south, north, west, east = location.raw["boundingbox"]

    for _ in range(10):
        lat = random.uniform(float(south), float(north))
        lon = random.uniform(float(west), float(east))

        result = geolocator.reverse(
            (lat, lon),
            addressdetails=True,
            exactly_one=True
        )

        if not result:
            continue

        address = result.raw.get("address", {})

        result_district = (
            address.get("neighbourhood")
            or address.get("suburb")
            or address.get("quarter")
            or address.get("district")
        )

        if result_district and result_district.lower() == district.lower():
            return result

    raise ValueError(f"Could not find an address in {district}.")


def build_customer(location, locale="pt_BR"):
    """Generate a customer from a real geographic address."""
    fake = Faker(locale)
    address = location.raw.get("address", {})

    return {
        "name": fake.name(),
        "email": fake.email(),
        "address": location.address,
        "zip": address.get("postcode"),
        "district": (
            address.get("neighbourhood")
            or address.get("suburb")
            or address.get("quarter")
            or address.get("district")
        ),
        "city": (
            address.get("city")
            or address.get("town")
            or address.get("village")
            or address.get("municipality")
        ),
        "state": address.get("state"),
        "country": address.get("country"),
        "latitude": location.latitude,
        "longitude": location.longitude,
    }


def create_customer(config):
    district = config.get("district")

    location = get_location(
        country=config.get("country", "Brazil"),
        state=config.get("state"),
        city=config.get("city"),
        district=district,
    )

    address = get_address(location, district)

    return build_customer(
        address,
        locale=config.get("locale", "pt_BR"),
    )


config = {
    "country": "Brazil",
    "state": "Parana",
    "city": "Curitiba",
    "district": "centro",
    "locale": "pt_BR",
}

customer = create_customer(config)

customer

In [0]:
config = {
    "country": "Brazil",
    "state": "Parana",
    "city": "Curitiba",
    "district": "Batel",
    "locale": "pt_BR",
}

customer = create_customer(config)

customer